In [6]:
### Filtering water pixels and then correcting fake NoData to real NoData --> Edited

import arcpy
import os
import time
from datetime import timedelta
from arcpy.sa import SetNull, Raster, IsNull

# -----------------------------
# Helpers
# -----------------------------
def log(msg):
    try:
        arcpy.AddMessage(msg)
    except Exception:
        print(msg)

def fmt(sec):
    return str(timedelta(seconds=int(max(0, sec))))

# -----------------------------
# PATHS
# -----------------------------
snap_raster = r"E:\Mosaic_Norge\snap_raster\snap_raster.tif"
water_mask  = r"E:\Vann_raster\Water_mask_10m_NODATA.tif"
in_dir  = r"E:\Mosaic_Norge\Test"
out_dir = r"E:\Mosaic_Norge\Final_rasts"

# -----------------------------
# Threshold rules
# Anything strictly > threshold becomes NoData
# (Use physically realistic maxima per raster type)
# -----------------------------
THRESHOLDS = {
    "dtm":    10000,   # meters; catches 1.5e15 and 3.4e38 safely
    "curv_":  1e6,     # curvature should not be huge; catches 1.5e15 safely
    "spi":    1e6,     # adjust once you confirm expected SPI range
    "tri":    1e6,     # TRI should not be billions
    "slope":  1000,    # slope is 0–90 (or 0–100); 1000 is ultra-safe
    "aspect": 360,     # aspect valid is -1..360 (we handle -1 separately)
}
DEFAULT_THRESHOLD = 1e14  # fallback if no key matched (kept high)

# Optional: DTM lower bound filter (disabled by default)
FILTER_DTM_LOWER = False
DTM_MIN_VALID = -500      # meters (set e.g. -500 if you want)

# -----------------------------
# ENV (stackable rasters)
# -----------------------------
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True
arcpy.env.snapRaster = snap_raster
arcpy.env.cellSize = snap_raster
arcpy.env.extent = snap_raster
arcpy.env.outputCoordinateSystem = snap_raster
arcpy.env.compression = "LZW"
arcpy.env.pyramid = "NONE"

# -----------------------------
# ETA (rough)
# -----------------------------
t0 = time.time()
rows = int(arcpy.management.GetRasterProperties(snap_raster, "ROWCOUNT").getOutput(0))
cols = int(arcpy.management.GetRasterProperties(snap_raster, "COLUMNCOUNT").getOutput(0))
cells = rows * cols
FAST, SLOW = 15_000_000, 3_000_000

log("--------------------------------------------------")
log("DL raster cleaning + masking")
log(f"Snap raster: {snap_raster}")
log(f"Grid size :  {rows} x {cols} = {cells:,} cells")
log(f"ETA per raster (rough): {fmt(cells/FAST)} to {fmt(cells/SLOW)}")
log("--------------------------------------------------")

# -----------------------------
# Setup
# -----------------------------
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

wm = Raster(water_mask)

rasters = [f for f in os.listdir(in_dir) if f.lower().endswith("_mosaic.tif")]
n = len(rasters)
if n == 0:
    raise RuntimeError("No rasters found to process.")

arcpy.SetProgressor("step", "Cleaning rasters for DL stacking...", 0, n, 1)

def pick_threshold(fname_lower: str) -> float:
    # first match wins
    for key, thr in THRESHOLDS.items():
        if key in fname_lower:
            return thr
    return DEFAULT_THRESHOLD

# -----------------------------
# Main loop
# -----------------------------
for i, fname in enumerate(rasters, 1):
    start = time.time()
    fname_l = fname.lower()

    in_path = os.path.join(in_dir, fname)
    out_name = fname.replace("_mosaic.tif", "_DLready.tif")
    out_path = os.path.join(out_dir, out_name)

    arcpy.SetProgressorLabel(f"[{i}/{n}] Processing {fname}")

    r = Raster(in_path)

    # 1) Remove water (mask: water=1, land=NoData)
    # Correct logic: wherever mask is NOT NoData => water => set output to NoData
    land = SetNull(~IsNull(wm), r)

    # 2) Convert fake NoData to real NoData using per-layer threshold
    thr = pick_threshold(fname_l)
    log(f"Using threshold for {fname}: values > {thr:g} become NoData")
    land = SetNull(land > thr, land)

    # Optional: DTM low-end filter
    if FILTER_DTM_LOWER and "dtm" in fname_l:
        land = SetNull(land < DTM_MIN_VALID, land)

    # 3) Aspect: set -1 to NoData
    if "aspect" in fname_l:
        land = SetNull(land == -1, land)

    # Write output safely
    if arcpy.Exists(out_path):
        arcpy.management.Delete(out_path)

    arcpy.management.CopyRaster(land, out_path, format="TIFF")

    elapsed = time.time() - start
    remaining = (n - i) * elapsed
    log(f"✔ {fname} | {fmt(elapsed)} | ETA left {fmt(remaining)}")

    arcpy.SetProgressorPosition()

# -----------------------------
# Finish
# -----------------------------
arcpy.ResetProgressor()
arcpy.CheckInExtension("Spatial")
log("")
log("✅ ALL DONE")
log(f"Outputs written to: {out_dir}")
